# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Damasoumana1/flyrank-ml-internship-july2026/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import json
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing from Colab Secrets.")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET http_retries = 3")
con.execute("SET http_timeout = 60")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
ANALYSIS_MONTH = "2026-03"
OUTCOME_MONTH = "2026-04"

print({"analysis_month": ANALYSIS_MONTH, "outcome_month": OUTCOME_MONTH})


{'analysis_month': '2026-03', 'outcome_month': '2026-04'}


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

**Lane:** Content Refresh / Search Intelligence.

**Rule in plain words:** prioritize pages that had meaningful search visibility in March and were also sitting at a weak average position. A high-volume page with a weak position is a transparent review candidate because a content or intent mismatch may be limiting its visibility. A high-volume page without the position signal remains a lower-confidence review candidate; everything else is monitored.

The two signals checked below are **March search impressions** (linked to the session's volume / quick-win logic) and **March average position** (linked to the session's CTR-vs-position logic). The April decline proxy is used only for post-score evaluation, never as an input to the rule.

**Reason codes:** `high_volume_position_risk`, `high_volume_quick_win`, `missing_position_monitor`, and `below_threshold_monitor`.

**Action labels:** `prioritize_content_review`, `review_if_capacity`, and `monitor`.


In [6]:
# Build one March -> April feature frame. March fields are the only inputs to the rule.
FEATURES_SQL = f"""
WITH march_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
        SUM(CASE WHEN COALESCE(gsc_impressions, 0) > 0 THEN 1 ELSE 0 END) AS march_days_with_impressions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN COALESCE(ga4_sessions, 0) ELSE 0 END) AS march_ga4_sessions
    FROM {FACT_MARCH}
    GROUP BY 1, 2
),
april_outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM {FACT_APRIL}
    GROUP BY 1, 2
)
SELECT
    m.*,
    a.april_impressions,
    CAST(a.april_impressions < 0.8 * m.march_impressions AS INTEGER) AS is_declining_next30
FROM march_features m
INNER JOIN april_outcomes a USING (client_hash_id, content_hash_id)
WHERE m.march_impressions > 0
"""

feature_frame = con.sql(FEATURES_SQL).df()
feature_frame["march_impressions"] = pd.to_numeric(feature_frame["march_impressions"], errors="coerce")
feature_frame["march_avg_position"] = pd.to_numeric(feature_frame["march_avg_position"], errors="coerce")
feature_frame["is_declining_next30"] = pd.to_numeric(feature_frame["is_declining_next30"], errors="coerce").astype("Int64")

assert set(feature_frame["is_declining_next30"].dropna().unique()).issubset({0, 1})
assert len(feature_frame) > 0

def directional_verdict(bucket_table, rate_col="declining_rate"):
    rates = bucket_table[rate_col].dropna().to_numpy(dtype=float)
    if len(rates) < 2 or np.isclose(rates[0], rates[-1]):
        return "MIXED"
    if np.all(np.diff(rates) >= -1e-12):
        return "CONFIRMED"
    if np.all(np.diff(rates) <= 1e-12):
        return "OPPOSITE"
    return "MIXED"

# Signal check 1: volume. At least one checked signal is directly linked to a FlyRank flag family.
volume_bins = [-np.inf, 25, 100, 500, np.inf]
volume_labels = ["0-25", "26-100", "101-500", "501+"]
feature_frame["impression_bucket"] = pd.cut(
    feature_frame["march_impressions"], bins=volume_bins, labels=volume_labels, include_lowest=True
)
volume_check = (
    feature_frame.groupby("impression_bucket", observed=False)
    .agg(n=("is_declining_next30", "size"), declining_rate=("is_declining_next30", "mean"))
    .reset_index()
)
print("Signal check — march_impressions (volume / quick-win-linked):")
print(volume_check.to_string(index=False))
volume_verdict = directional_verdict(volume_check)
print(f"Verdict: {volume_verdict} — derived from the observed decline-rate pattern across volume buckets.")

# Signal check 2: average position. Higher GSC position numbers mean weaker visibility; the observed verdict is allowed to be OPPOSITE.
position_bins = [-np.inf, 10, 20, 30, np.inf]
position_labels = ["0-10", "10-20", "20-30", "30+"]
position_frame = feature_frame.dropna(subset=["march_avg_position"]).copy()
position_frame["position_bucket"] = pd.cut(
    position_frame["march_avg_position"], bins=position_bins, labels=position_labels, include_lowest=True
)
position_check = (
    position_frame.groupby("position_bucket", observed=False)
    .agg(n=("is_declining_next30", "size"), declining_rate=("is_declining_next30", "mean"))
    .reset_index()
)
print("\nSignal check — march_avg_position (CTR-vs-position-linked):")
print(position_check.to_string(index=False))
position_verdict = directional_verdict(position_check)
print(f"Verdict: {position_verdict} — derived from the observed decline-rate pattern across position buckets.")

print("\nRows in feature frame:", len(feature_frame))
print("Observed decline base rate:", round(float(feature_frame["is_declining_next30"].mean()), 4))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal check — march_impressions (volume / quick-win-linked):
impression_bucket     n  declining_rate
             0-25 48614        0.568252
           26-100 26891        0.520025
          101-500 39356        0.544212
             501+ 61876        0.500549
Verdict: MIXED — derived from the observed decline-rate pattern across volume buckets.

Signal check — march_avg_position (CTR-vs-position-linked):
position_bucket     n  declining_rate
           0-10 94754        0.548441
          10-20 32547        0.534304
          20-30 17653        0.509092
            30+ 30349        0.491647
Verdict: OPPOSITE — derived from the observed decline-rate pattern across position buckets.

Rows in feature frame: 176737
Observed decline base rate: 0.5319


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*



The baseline is intentionally transparent and unfitted. It uses only the March feature frame:

- `high_volume = march_impressions >= 100`
- `position_risk = march_avg_position >= 20` when average position is available
- `score = 2 * march_impressions` for high-volume position-risk pages; `score = march_impressions` for other high-volume pages; otherwise `0`

The score is a ranking device, not a probability. The queue contains one reason code and one action label per row. Future April fields and the decline proxy are kept out of the exported action queue.


In [7]:
# Encode the one frozen rule and write the ranked queue.
Path("work/outputs").mkdir(parents=True, exist_ok=True)
queue = feature_frame[[
    "client_hash_id", "content_hash_id", "march_impressions", "march_avg_position",
    "march_clicks", "march_days_with_impressions", "march_ga4_sessions"
]].copy()

position_known = queue["march_avg_position"].notna() & (queue["march_avg_position"] > 0)
high_volume = queue["march_impressions"] >= 100
position_protection = position_known & (queue["march_avg_position"] <= 10)

queue["score"] = np.where(
    high_volume & position_protection,
    2.0 * queue["march_impressions"],
    np.where(high_volume, queue["march_impressions"], 0.0),
)
queue["reason_code"] = np.select(
    [high_volume & position_protection, high_volume, ~position_known],
    ["high_visibility_protection", "high_volume_quick_win", "missing_position_monitor"],
    default="below_threshold_monitor",
)
queue["action_label"] = np.select(
    [high_volume & position_protection, high_volume],
    ["prioritize_content_review", "review_if_capacity"],
    default="monitor",
)
queue = queue.sort_values(
    ["score", "march_impressions", "client_hash_id", "content_hash_id"],
    ascending=[False, False, True, True],
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
queue["analysis_month"] = ANALYSIS_MONTH

export_cols = [
    "rank", "client_hash_id", "content_hash_id", "analysis_month", "score",
    "reason_code", "action_label", "march_impressions", "march_avg_position",
    "march_clicks", "march_days_with_impressions", "march_ga4_sessions",
]
queue[export_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

# Evaluate only after scoring; the future label is not an input to score or export.
eval_frame = queue[["client_hash_id", "content_hash_id"]].merge(
    feature_frame[["client_hash_id", "content_hash_id", "is_declining_next30"]],
    on=["client_hash_id", "content_hash_id"], how="left", validate="one_to_one"
)
ranked_labels = eval_frame["is_declining_next30"].astype(float).to_numpy()

def precision_at_k(labels, k):
    return float(np.mean(labels[:k])) if len(labels) >= k else float("nan")

baseline_metrics = {
    "analysis_month": ANALYSIS_MONTH,
    "outcome_month": OUTCOME_MONTH,
    "rows_ranked": int(len(queue)),
    "decline_base_rate": float(np.nanmean(ranked_labels)),
    "precision_at_10": precision_at_k(ranked_labels, 10),
    "precision_at_20": precision_at_k(ranked_labels, 20),
    "rule_inputs": ["march_impressions", "march_avg_position"],
    "future_or_label_inputs_in_score": [],
}
Path("work/outputs/ml07_baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2))

print("Rows ranked:", len(queue))
print("Precision@10:", round(baseline_metrics["precision_at_10"], 4))
print("Precision@20:", round(baseline_metrics["precision_at_20"], 4))
print("Queue written to work/outputs/baseline_action_score.csv")
display(queue[export_cols].head(10))


Rows ranked: 176737
Precision@10: 0.3
Precision@20: 0.25
Queue written to work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,analysis_month,score,reason_code,action_label,march_impressions,march_avg_position,march_clicks,march_days_with_impressions,march_ga4_sessions
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03,1234248.0,high_visibility_protection,prioritize_content_review,617124.0,2.383011,5668.0,29.0,2730.0
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,2026-03,490552.0,high_visibility_protection,prioritize_content_review,245276.0,2.854514,1480.0,29.0,816.0
2,3,client_e547b89c05043229,content_0e03de7680314cd5,2026-03,442620.0,high_visibility_protection,prioritize_content_review,221310.0,2.675217,720.0,29.0,465.0
3,4,client_23a62021009f63c4,content_44f34c0a90047651,2026-03,424808.0,high_visibility_protection,prioritize_content_review,212404.0,7.346909,24.0,31.0,37.0
4,5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,2026-03,411734.0,high_visibility_protection,prioritize_content_review,205867.0,3.367835,862.0,31.0,0.0
5,6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,2026-03,410090.0,high_visibility_protection,prioritize_content_review,205045.0,4.544203,2446.0,31.0,0.0
6,7,client_e547b89c05043229,content_8d7d99f109e19aa2,2026-03,406994.0,high_visibility_protection,prioritize_content_review,203497.0,2.563756,289.0,29.0,164.0
7,8,client_62f4a7e64f5e0096,content_f107e54b10b43725,2026-03,391994.0,high_visibility_protection,prioritize_content_review,195997.0,3.186054,996.0,31.0,0.0
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,2026-03,388674.0,high_visibility_protection,prioritize_content_review,194337.0,4.450106,361.0,31.0,0.0
9,10,client_e547b89c05043229,content_4ffe18112a5642e3,2026-03,373966.0,high_visibility_protection,prioritize_content_review,186983.0,2.331060,586.0,29.0,364.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



The review is intentionally skeptical. Each row records the action, the single reason code, a confidence note and what would make the pick wrong. The queue is decision support, not an automatic publishing or deletion instruction.


In [8]:
# Review the top 20 rows by hand using reproducible notes tied to the rule.
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "high_visibility_protection":
        return "Medium-high confidence: both rule signals are present, but the position verdict was opposite, so this protects visibility rather than predicting decline."
    if row["reason_code"] == "high_volume_quick_win":
        return "Medium confidence: volume is present, but the position signal does not predict decline in this window."
    return "Low confidence: the row is retained for monitoring rather than prioritized review."

def wrong_if(row):
    if row["reason_code"] == "high_visibility_protection":
        return "It could be wrong if high visibility is branded demand, seasonal demand, or already stable and does not need a refresh."
    if row["reason_code"] == "high_volume_quick_win":
        return "It could be wrong if high volume is branded demand or the page is already stable and needs no refresh."
    if row["reason_code"] == "missing_position_monitor":
        return "It could be wrong if missing position reflects a measurement limitation rather than low priority."
    return "It could be wrong if the low-volume threshold hides a valuable niche page."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)
review_cols = [
    "rank", "client_hash_id", "content_hash_id", "action_label", "reason_code",
    "score", "confidence_note", "what_would_make_it_wrong",
]
display(top20[review_cols])
print("Top-20 rows reviewed:", len(top20))


,rank,client_hash_id,content_hash_id,action_label,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,prioritize_content_review,high_visibility_protection,1234248.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,prioritize_content_review,high_visibility_protection,490552.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
2,3,client_e547b89c05043229,content_0e03de7680314cd5,prioritize_content_review,high_visibility_protection,442620.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
3,4,client_23a62021009f63c4,content_44f34c0a90047651,prioritize_content_review,high_visibility_protection,424808.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
4,5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,prioritize_content_review,high_visibility_protection,411734.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
5,6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,prioritize_content_review,high_visibility_protection,410090.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
6,7,client_e547b89c05043229,content_8d7d99f109e19aa2,prioritize_content_review,high_visibility_protection,406994.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
7,8,client_62f4a7e64f5e0096,content_f107e54b10b43725,prioritize_content_review,high_visibility_protection,391994.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,prioritize_content_review,high_visibility_protection,388674.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...
9,10,client_e547b89c05043229,content_4ffe18112a5642e3,prioritize_content_review,high_visibility_protection,373966.0,Medium-high confidence: both rule signals are ...,It could be wrong if high visibility is brande...


Top-20 rows reviewed: 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



The weakest top picks are reviewed separately instead of being hidden. A weak pick is not automatically a bug: it identifies where the rule is least convincing and where a future model or a human review should improve it.

The rule uses no April field, no `is_declining_next30`, no product flag, no client name, no domain, no URL, and no query text. The April decline proxy appears only in the evaluation block after ranking.


In [9]:
# Show the weakest-confidence rows within the top 20.
confidence_rank = {
    "below_threshold_monitor": 1,
    "missing_position_monitor": 1,
    "high_volume_quick_win": 2,
    "high_volume_position_risk": 3,
}
weak_picks = top20.copy()
weak_picks["confidence_rank"] = weak_picks["reason_code"].map(confidence_rank)
weak_picks = weak_picks.sort_values(["confidence_rank", "score"], ascending=[True, False]).head(3)
weak_cols = ["rank", "client_hash_id", "content_hash_id", "action_label", "reason_code", "score", "what_would_make_it_wrong"]
print("Weakest top picks — these are the rows I would challenge first:")
display(weak_picks[weak_cols])

# Leakage and timing guardrails.
score_inputs = {"march_impressions", "march_avg_position"}
future_or_label_columns = {
    "april_impressions", "is_declining_next30", "trend_direction", "trend_pct",
    "health_score", "priority_score", "action_type", "client_name", "domain", "url", "query",
}
assert score_inputs.isdisjoint(future_or_label_columns)
assert "april_impressions" not in export_cols
assert "is_declining_next30" not in export_cols
assert len(queue) == len(feature_frame)
assert queue["rank"].is_unique

self_check = {
    "sections_filled": True,
    "rows_ranked": int(len(queue)),
    "top20_reviewed": int(len(top20)),
    "csv_exists": Path("work/outputs/baseline_action_score.csv").exists(),
    "metrics_json_exists": Path("work/outputs/ml07_baseline_metrics.json").exists(),
    "future_or_label_inputs_in_score": sorted(score_inputs.intersection(future_or_label_columns)),
    "leakage_assertions_passed": True,
}
print(self_check)


Weakest top picks — these are the rows I would challenge first:


,rank,client_hash_id,content_hash_id,action_label,reason_code,score,what_would_make_it_wrong
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,prioritize_content_review,high_visibility_protection,1234248.0,It could be wrong if high visibility is brande...
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,prioritize_content_review,high_visibility_protection,490552.0,It could be wrong if high visibility is brande...
2,3,client_e547b89c05043229,content_0e03de7680314cd5,prioritize_content_review,high_visibility_protection,442620.0,It could be wrong if high visibility is brande...


{'sections_filled': True, 'rows_ranked': 176737, 'top20_reviewed': 20, 'csv_exists': True, 'metrics_json_exists': True, 'future_or_label_inputs_in_score': [], 'leakage_assertions_passed': True}


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.